Exercice 1 — LoRALayer

Cellule 1 — Installation

In [10]:
!pip uninstall peft torchao -y --quiet
!pip install torchao==0.16.0 --quiet
!pip install peft transformers accelerate datasets --quiet

print("Installation terminée !")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 20.1 MB/s eta 0:00:00
Installation terminée !


In [11]:
import torchao
import peft
print(f"torchao : {torchao.__version__}")
print(f"peft    : {peft.__version__}")

# Test rapide
from peft import LoraConfig, get_peft_model
print("Import OK !")

torchao : 0.16.0
peft    : 0.19.1
Import OK !


Cellule 2 — Charger le modèle et le tokenizer

In [12]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name       = "bigscience/bloomz-560m"
tokenizer        = AutoTokenizer.from_pretrained(model_name)
foundation_model = AutoModelForCausalLM.from_pretrained(model_name)

print(f"Modèle chargé : {model_name}")
print(f"Nombre de paramètres : {foundation_model.num_parameters():,}")

Modèle chargé : bigscience/bloomz-560m
Nombre de paramètres : 559,214,592


Cellule 3 — Charger et préparer le dataset

In [13]:
from datasets import load_dataset

data = load_dataset("Abirate/english_quotes", split="train")
data = data.shuffle(seed=42).select(range(len(data) // 10))
data = data.map(
    lambda samples: tokenizer(samples["quote"]),
    batched=True
)

train_sample = data.select(range(5))
display(train_sample)

print(f"\nNombre total d'exemples : {len(data)}")
print(f"Colonnes                : {data.column_names}")
print(f"Exemple de citation     : {data[0]['quote']}")

Dataset({
    features: ['quote', 'author', 'tags', 'input_ids', 'attention_mask'],
    num_rows: 5
})


Nombre total d'exemples : 250
Colonnes                : ['quote', 'author', 'tags', 'input_ids', 'attention_mask']
Exemple de citation     : “I don't mind making jokes, but I don't want to look like one.”


Cellule 4 — Configurer LoRA

In [14]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=1,
    lora_alpha=1,
    target_modules=["query_key_value"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

peft_model = get_peft_model(foundation_model, lora_config)
peft_model.print_trainable_parameters()

trainable params: 98,304 || all params: 559,312,896 || trainable%: 0.0176


Cellule 5 — Entraîner le modèle

In [15]:
import transformers
from transformers import TrainingArguments, Trainer
import os

# Chemin corrigé pour Colab
output_directory = "/content/peft_lab_outputs"
os.makedirs(output_directory, exist_ok=True)

training_args = TrainingArguments(
    report_to="none",
    output_dir=output_directory,
    auto_find_batch_size=True,
    learning_rate=3e-2,
    num_train_epochs=5,
    no_cuda=True  # ← remplace use_cpu=True qui est déprécié
)

trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=train_sample,  # ← corrigé (était "e" dans le hint)
    data_collator=transformers.DataCollatorForLanguageModeling(
        tokenizer,
        mlm=False
    )
)

trainer.train()
print("Entraînement terminé !")

/usr/local/lib/python3.12/dist-packages/transformers/training_args.py:1540: FutureWarning: using `no_cuda` is deprecated and will be removed in version 5.0 of 🤗 Transformers. Use `use_cpu` instead
  warnings.warn(


Step,Training Loss


Entraînement terminé !


Cellule 6 — Sauvegarder le modèle

In [16]:
import time

time_now        = int(time.time())
peft_model_path = os.path.join(output_directory, f"peft_model_{time_now}")

trainer.model.save_pretrained(peft_model_path)

print(f"Modèle sauvegardé dans : {peft_model_path}")
print("Fichiers sauvegardés :")
for f in os.listdir(peft_model_path):
    print(f"  {f}")

Modèle sauvegardé dans : /content/peft_lab_outputs/peft_model_1782438989
Fichiers sauvegardés :
  README.md
  adapter_config.json
  adapter_model.safetensors


Cellule 7 — Charger et générer du texte

In [17]:
from peft import PeftModel

loaded_model = PeftModel.from_pretrained(
    foundation_model,
    peft_model_path,
    is_trainable=False
)

print(f"Modèle chargé depuis : {peft_model_path}")

inputs = tokenizer("Two things are infinite: ", return_tensors="pt")

outputs = loaded_model.generate(
    input_ids=inputs["input_ids"],
    max_new_tokens=50,
    do_sample=True,          # échantillonnage activé
    temperature=0.8,
    pad_token_id=tokenizer.eos_token_id
)

print("\n=== Texte généré ===")
print(tokenizer.batch_decode(outputs, skip_special_tokens=True)[0])

/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Modèle chargé depuis : /content/peft_lab_outputs/peft_model_1782438989

=== Texte généré ===
Two things are infinite:  we are not in the world but we are (‘and, yes, we are; but if we are  and are, what are we to ourselves:) but what we are not: but what they say is nothing more than a play on


Cellule 8 — Comparer avant/après fine-tuning

In [18]:
inputs = tokenizer("Two things are infinite: ", return_tensors="pt")

# Modèle original
outputs_original = foundation_model.generate(
    input_ids=inputs["input_ids"],
    max_new_tokens=50,
    do_sample=True,
    temperature=0.8,
    pad_token_id=tokenizer.eos_token_id
)

# Modèle fine-tuné LoRA
outputs_lora = loaded_model.generate(
    input_ids=inputs["input_ids"],
    max_new_tokens=50,
    do_sample=True,
    temperature=0.8,
    pad_token_id=tokenizer.eos_token_id
)

print("=== Modèle ORIGINAL ===")
print(tokenizer.batch_decode(outputs_original, skip_special_tokens=True)[0])

print("\n=== Modèle FINE-TUNÉ avec LoRA ===")
print(tokenizer.batch_decode(outputs_lora, skip_special_tokens=True)[0])

=== Modèle ORIGINAL ===
Two things are infinite:  ‘I am as god, but, not the ‘me,’’ ‘the’, but in myself not the God,'”””…‘There is the whole world in you” (Ephelians 5,8).”

=== Modèle FINE-TUNÉ avec LoRA ===
Two things are infinite:  everything is not for me.””””And a moment,”” he added, is only such a moment.”””” (p. 43)”’” The best thing that we can be are ourselves,” says the philosopher
